# 4. MLOps Deployment: MLflow, GCS, and Vertex AI
This notebook documents the transfer of model artifacts to Google Cloud Storage and the deployment of the model as a live endpoint on Vertex AI.

## 1. Import Required Libraries
We import libraries for model packaging, cloud storage, and deployment.

In [ ]:
import joblib
import os
# For Google Cloud Storage and Vertex AI, use the google-cloud-storage and google-cloud-aiplatform packages
# !pip install google-cloud-storage google-cloud-aiplatform
from google.cloud import storage
from google.cloud import aiplatform

## 2. Package and Save the Final Model
We combine the preprocessing pipeline and the best classification model into a single pipeline and save it as model.joblib.

In [ ]:
# Assume cls_pipeline is the best model from supervised modeling notebook
# Save the model pipeline
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(cls_pipeline, '../artifacts/model.joblib')
print('Model pipeline saved to ../artifacts/model.joblib')

## 3. Prepare requirements.txt
We list all required Python packages for deployment.

## 4. Upload Artifacts to Google Cloud Storage (GCS)
We upload model.joblib and requirements.txt to a GCS bucket for deployment.

In [ ]:
# Set your GCS bucket name
bucket_name = 'your-gcs-bucket-name'

# Initialize GCS client
client = storage.Client()
bucket = client.bucket(bucket_name)

# Upload model.joblib
model_blob = bucket.blob('model.joblib')
model_blob.upload_from_filename('../artifacts/model.joblib')
print('Uploaded model.joblib to GCS')

# Upload requirements.txt
req_blob = bucket.blob('requirements.txt')
req_blob.upload_from_filename('../artifacts/requirements.txt')
print('Uploaded requirements.txt to GCS')

## 5. Deploy Model to Vertex AI Endpoint
We import the model from GCS, select the pre-built Scikit-learn container, and deploy as a live endpoint.

In [ ]:
# Set your project and region
project = 'your-gcp-project-id'
location = 'us-central1'

# Initialize Vertex AI
aiplatform.init(project=project, location=location)

# Import model from GCS
model = aiplatform.Model.upload(
    display_name='AuraCart Customer Segment Model',
    artifact_uri=f'gs://{bucket_name}/model.joblib',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest',
    requirements=['gs://{}/requirements.txt'.format(bucket_name)]
)

# Deploy to endpoint
endpoint = model.deploy(machine_type='n1-standard-2')
print('Model deployed to Vertex AI endpoint:', endpoint.resource_name)

## 6. Test the Live Endpoint
Send a sample JSON payload to the endpoint and display the prediction response.

In [ ]:
# Example: send a test request (replace with your endpoint details)
from google.cloud.aiplatform.gapic.prediction_service_client import PredictionServiceClient
from google.cloud.aiplatform.gapic.schema import predict

endpoint_id = 'your-endpoint-id'
instance = {
    # Fill with a sample input matching the model's expected features
}

client = PredictionServiceClient(client_options={"api_endpoint": f"{location}-aiplatform.googleapis.com"})
response = client.predict(
    endpoint=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}",
    instances=[instance],
)
print('Prediction response:', response)